In [ ]:
"""
frame_alignment_vid.ipynb

generate a video to confirm that the alignment is correct

Author: Stellina X. Ao
Created: 2026-11-11
Last Modified: 2026-11-11
Python Version: 3.11.14
"""

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
from core.data import default_sessions

subj_id = "MR82"
sess_id = default_sessions[subj_id]

In [ ]:
from sg.models import make_tre, Encoder

encoder_svd = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    add_svd=True,
    num_svd=20,
    stepsize_s=0.1,
)
encoder_svd.verify()

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    tv_keys=["response", "rewarded", "response_prev", "rewarded_prev"],
    add_svd=False,
    stepsize_s=0.1,
)
encoder.verify()

### alignment

In [ ]:
from utils.paths import DATA_DIR
import cv2

vid_path = (
    DATA_DIR
    / subj_id
    / sess_id
    / "MR82_DynamicForaging_20251027_152036_cam1_run000_00000000.avi"
)

cap = cv2.VideoCapture(str(vid_path))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()
print(n_frames)

In [ ]:
import cv2


def put_text(frame, text, org, scale, color, thickness):
    font = cv2.FONT_HERSHEY_DUPLEX
    cv2.putText(frame, text, org, font, scale, (0, 0, 0), thickness + 2, cv2.LINE_AA)
    cv2.putText(frame, text, org, font, scale, color, thickness, cv2.LINE_AA)


def write_trial_video(
    vid_path,
    frame_trial_idxs,
    trial_idxs,
    out_path,
    fourcc="mp4v",
    flash_s=0.25,
    flash_color=(0, 0, 255),  # BGR
    text_color=(255, 255, 255),
    font_scale=1.0,
    thickness=2,
):
    cap = cv2.VideoCapture(str(vid_path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(
        str(out_path), cv2.VideoWriter_fourcc(*fourcc), fps, (w, h)
    )

    flash_frames = max(1, round(flash_s * fps))
    border = max(4, w // 80)

    try:
        for t in trial_idxs:
            start, end = (int(i) for i in frame_trial_idxs[t])
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            for f in range(start, end):
                ok, frame = cap.read()
                if not ok:
                    print(f"trial {t}: read failed at frame {f}")
                    break

                if f - start < flash_frames:
                    # rectangle is centred on the frame edge, so double the thickness
                    cv2.rectangle(
                        frame, (0, 0), (w - 1, h - 1), flash_color, 2 * border
                    )

                put_text(
                    frame,
                    f"trial {t}   {(f - start) / fps:.2f} s",
                    (15, int(40 * font_scale)),
                    font_scale,
                    text_color,
                    thickness,
                )
                writer.write(frame)
    finally:
        cap.release()
        writer.release()

In [ ]:
import numpy as np
from utils.paths import DATA_DIR
from core.data import get_event_trial_idxs

trial_data = encoder.trial_data
session_data = encoder.session_data
svd_stepsize = None


trial_start = session_data["events"]["event_timestamps"][0]
movie_frame = session_data["events"]["event_timestamps"][1]

movements = np.load(
    DATA_DIR
    / subj_id
    / sess_id
    / f"{subj_id}_DynamicForaging_{sess_id}_cam1_run000_00000000.npy",
    allow_pickle=True,
).item()

frame_trial_idxs = get_event_trial_idxs(
    trial_start,
    trial_data,
    movie_frame,
    svd_stepsize=svd_stepsize,
    tpre=0.5,
    tpost=2.0,
    alignment="trial_start",
    full_trial=False,
)

In [ ]:
import cv2
from utils.paths import FIGURES_DIR

out_path = FIGURES_DIR / "svds" / "explore" / "spout_movement_anticipatory.mp4"
fourcc = "mp4v"
trial_idxs = [20, 22, 54, 67, 101]  # first three and last three

write_trial_video(
    vid_path=vid_path,
    frame_trial_idxs=frame_trial_idxs,
    trial_idxs=trial_idxs,
    out_path=out_path,
)